In [1]:
from langchain_groq import ChatGroq
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END
import base64
import cv2
from typing_extensions import TypedDict, Any
import os
import matplotlib.pyplot as plt
import time
from inference_sdk import InferenceHTTPClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import json
from pydantic import BaseModel
import numpy as np
from io import BytesIO
from PIL import Image

C:\Users\rohit\miniconda3\envs\Genai\lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
C:\Users\rohit\miniconda3\envs\Genai\lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


# =========================================================================

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    ElementNotInteractableException,
    StaleElementReferenceException
)
def safe_click(driver, xpath):

    wait = WebDriverWait(driver, 10)

    element = wait.until(
        EC.presence_of_element_located((By.XPATH, xpath))
    )

    # Scroll to center
    driver.execute_script("""
        arguments[0].scrollIntoView({
            block:'center',
            inline:'center'
        });
    """, element)

    time.sleep(0.4)

    try:
        wait.until(
            EC.element_to_be_clickable((By.XPATH, xpath))
        )

        element.click()
        return

    except ElementClickInterceptedException:

        print("Normal click intercepted.")

    except ElementNotInteractableException:

        print("Element not interactable.")

    # Try ActionChains
    try:

        ActionChains(driver)\
            .move_to_element(element)\
            .pause(0.2)\
            .click()\
            .perform()

        return

    except Exception:

        pass

    # Last fallback
    driver.execute_script(
        "arguments[0].click();",
        element
    )


In [5]:
def get_interactive_elements(driver):
    elements = driver.execute_script("""
function getXPath(el) {

    if (el.id)
        return `//*[@id="${el.id}"]`;

    let path = [];

    while (el && el.nodeType === 1) {

        let index = 1;

        let sibling = el.previousElementSibling;

        while (sibling) {
            if (sibling.tagName === el.tagName)
                index++;

            sibling = sibling.previousElementSibling;
        }

        path.unshift(
            el.tagName.toLowerCase() +
            "[" + index + "]"
        );

        el = el.parentElement;
    }

    return "/" + path.join("/");
}


let selectors = `
button,
a,
input,
textarea,
select,
[role="button"],
[role="link"],
[onclick],
[tabindex]
`;

let nodes = document.querySelectorAll(selectors);

let results = [];

nodes.forEach(el => {

    let rect = el.getBoundingClientRect();

    let visible =
        rect.width > 0 &&
        rect.height > 0 &&
        window.getComputedStyle(el).display !== "none" &&
        window.getComputedStyle(el).visibility !== "hidden";

    if (!visible)
        return;

    let text =
        el.innerText ||
        el.value ||
        el.placeholder ||
        el.getAttribute("aria-label") ||
        "";

    results.push({

        tag: el.tagName.toLowerCase(),

        text: text.trim(),

        aria_label:
            el.getAttribute("aria-label") || "",

        placeholder:
            el.getAttribute("placeholder") || "",

        name:
            el.getAttribute("name") || "",

        type:
            el.getAttribute("type") || "",

        xpath:
            getXPath(el)
    });

});

return results;
""")


    for idx, ele in enumerate(elements):
        ele["index"] = idx

    return(json.loads(json.dumps(elements, indent=2)))

In [6]:
# def encode_image(image_path):
#     with open(image_path, "rb") as image_file:
#         return base64.b64encode(image_file.read()).decode('utf-8')


def encode_image(image_path, max_dimension=448, quality=85):
    """
    Resizes an image maintaining aspect ratio and encodes it to Base64.
    """
    with Image.open(image_path) as img:
        # Convert RGBA/Palette to RGB to ensure JPEG compatibility
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
            
        # Calculate new dimensions keeping the aspect ratio
        img.thumbnail((max_dimension, max_dimension))
        
        # Save to memory as a compressed JPEG
        buffered = BytesIO()
        img.save(buffered, format="JPEG", quality=quality)
        
        # Encode to Base64 string for Groq
        img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
        return f"data:image/jpeg;base64,{img_str}"

In [7]:
from pydantic import Field
class Obstacle_handler(TypedDict):
    x: int
    y: int
    image_path: str
    driver: Any
    to_be_found: str
    button_selector_vlm_model: Any

class Structure_button(BaseModel):
    x: int=Field(description="The x co ordinate of the button")
    y: int=Field(description="The y co ordinate of the button")  


def initial_state(state: Obstacle_handler)->Obstacle_handler:
    prompt=ChatPromptTemplate.from_messages([(
        "system", """You are a intelligent button locator in the given image
        Your task is to generate the x and y co ordinates of the button the user wants to click
        
        The user would be providing the image along with the summarized text on what button has to be clicked.
        Based on the interpretation, you have to return the exact co ordinates on where the button is present.
        
        No extra explanation is needed."""
    ),("human", [{"type":"text", "text": "{to_be_found}"}, 
                {"type": "image_url", "image_url": {"url":"{base64_image}"}}])])
    structured_vlm_model=state["button_selector_vlm_model"].with_structured_output(Structure_button)
    base_64_image=encode_image(state["image_path"])

    chain = prompt | structured_vlm_model

    co_output_text=chain.invoke({
        "to_be_found": state["to_be_found"],
        "base64_image": base_64_image
    })

    state["x"]=co_output_text.x
    state["y"]=co_output_text.y
    return state


def secondary_state(state: Obstacle_handler)->Obstacle_handler:
    driver = state["driver"]

    x=state["x"]
    y=state["y"]
    dpr = driver.execute_script("return window.devicePixelRatio")
    x, y = x / dpr, y / dpr

    driver.execute_script("""
        let ele = document.elementFromPoint(arguments[0], arguments[1]);
        if (!ele) return;

        // Walk up to find the nearest clickable ancestor
        let target = ele;
        while (target && target !== document.body) {
            const tag = target.tagName.toLowerCase();
            const role = target.getAttribute('role') || '';
            if (tag === 'button' || tag === 'a' || tag === 'input'
                || role === 'button' || role === 'link'
                || target.onclick || target.getAttribute('onclick')) {
                break;
            }
            target = target.parentElement;
        }
        target = target || ele;

        // Dispatch full event sequence for reliable handler firing
        ['mousedown', 'mouseup', 'click'].forEach(type => {
            target.dispatchEvent(new MouseEvent(type, {
                bubbles: true, cancelable: true,
                clientX: arguments[0], clientY: arguments[1]
            }));
        });
    """, x, y)
    import time
    time.sleep(1.5)
    return state
graph=StateGraph(Obstacle_handler)

graph.add_node("init_state", initial_state)
graph.add_node("second_state", secondary_state)

graph.set_entry_point("init_state")
graph.add_edge("init_state", "second_state")
graph.add_edge("second_state", END)

app=graph.compile()
    

In [8]:
from selenium.webdriver.chrome.service import Service

In [9]:
import json
import hashlib
import time
import re
from typing import Any, Dict, List, Optional
from pydantic import Field, BaseModel, RootModel

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
from selenium.webdriver.common.keys import Keys
from langchain_core.tools import tool
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

_EXTRACT_JS = r"""
return (function (onlyVisible, includeDisabled) {

  function getXPath(el) {
    if (el.id) return '//*[@id="' + el.id + '"]';
    let path = [];
    while (el && el.nodeType === 1) {
      let index = 1;
      let sibling = el.previousElementSibling;
      while (sibling) {
        if (sibling.tagName === el.tagName) index++;
        sibling = sibling.previousElementSibling;
      }
      path.unshift(el.tagName.toLowerCase() + '[' + index + ']');
      el = el.parentElement;
    }
    return '/' + path.join('/');
  }

  function isVisible(el) {
    const rect = el.getBoundingClientRect();
    const style = window.getComputedStyle(el);
    return rect.width > 0 && rect.height > 0 &&
           style.display !== 'none' &&
           style.visibility !== 'hidden' &&
           style.opacity !== '0';
  }

  function cleanText(t) {
    return (t || '').replace(/\s+/g, ' ').trim();
  }

  function textOf(node) {
    return cleanText(node.innerText || node.textContent);
  }

  function resolveLabel(el) {
    if (el.id) {
      const lbl = document.querySelector('label[for="' + CSS.escape(el.id) + '"]');
      if (lbl && textOf(lbl)) return textOf(lbl);
    }
    const wrapLbl = el.closest('label');
    if (wrapLbl && textOf(wrapLbl)) return textOf(wrapLbl);

    if (el.getAttribute('aria-label')) return cleanText(el.getAttribute('aria-label'));

    const labelledBy = el.getAttribute('aria-labelledby');
    if (labelledBy) {
      const txt = labelledBy.split(' ')
        .map(id => { const r = document.getElementById(id); return r ? textOf(r) : ''; })
        .filter(Boolean).join(' ');
      if (txt) return txt;
    }

    // Plain inline text immediately after the element, e.g.
    // <input type="checkbox" ...> I agree to terms
    const nextText = el.nextSibling && el.nextSibling.nodeType === 3 ? cleanText(el.nextSibling.textContent) : '';
    if (nextText && nextText.length < 80) return nextText;

    // <input ...><label>Text</label>  — label with no `for`, sitting right after the field.
    // Only trust it if it isn't already bound (via `for`) to some OTHER element.
    const nextEl = el.nextElementSibling;
    if (nextEl && nextEl.tagName === 'LABEL' && !nextEl.hasAttribute('for') && textOf(nextEl)) {
      return textOf(nextEl);
    }
    // <label>Text</label><input ...> — same idea, label comes first.
    const prevEl = el.previousElementSibling;
    if (prevEl && prevEl.tagName === 'LABEL' && !prevEl.hasAttribute('for') && textOf(prevEl)) {
      return textOf(prevEl);
    }

    if (el.getAttribute('placeholder')) return cleanText(el.getAttribute('placeholder'));
    if (el.title) return cleanText(el.title);

    // Last resort: whatever descriptive text is sitting near the element in
    // the DOM (previous siblings, table cells, ancestor wrappers).
    const near = nearbyText(el);
    if (near) return near;

    if (el.name) return cleanText(el.name.replace(/[_\-]+/g, ' ').replace(/([a-z])([A-Z])/g, '$1 $2'));
    return '';
  }

  const FIELD_TAGS = ['INPUT', 'SELECT', 'TEXTAREA', 'BUTTON'];

  function isFieldNode(n) {
    return n && n.nodeType === 1 && (FIELD_TAGS.includes(n.tagName) || n.isContentEditable === true);
  }

  // Text belonging directly to `node` — its own text nodes plus the visible
  // text of non-field child elements — but NOT text pulled from a nested
  // input/select/etc, so we never absorb a sibling field's typed value.
  function ownText(node) {
    let out = '';
    node.childNodes.forEach(c => {
      if (c.nodeType === 3) {
        out += ' ' + c.textContent;
      } else if (c.nodeType === 1 && !isFieldNode(c)) {
        out += ' ' + (c.innerText || c.textContent || '');
      }
    });
    return cleanText(out);
  }

  // Fallback used when no explicit label/name/placeholder/aria-* is present.
  // Looks at text physically near the element: preceding siblings, the
  // preceding cell in a table row (e.g. <tr><td>Email</td><td><input></td></tr>),
  // and a few levels of ancestor wrappers (common in div-based form layouts).
  // Always skips other form controls and `for`-bound labels so we never leak
  // a neighboring field's text or value.
  function nearbyText(el) {
    function usable(t) {
      return t && t.length > 0 && t.length < 100;
    }

    // 1) Table layout: label lives in an earlier <td>/<th> of the same row.
    const cell = el.closest('td, th');
    if (cell) {
      let prev = cell.previousElementSibling;
      while (prev) {
        const t = textOf(prev);
        if (usable(t)) return t;
        prev = prev.previousElementSibling;
      }
    }

    // 2) Preceding sibling elements at the same DOM level.
    let sib = el.previousElementSibling;
    let hops = 0;
    while (sib && hops < 4) {
      if (!isFieldNode(sib) && !(sib.tagName === 'LABEL' && sib.hasAttribute('for'))) {
        const t = textOf(sib);
        if (usable(t)) return t;
      }
      sib = sib.previousElementSibling;
      hops++;
    }

    // 3) Walk up ancestor wrappers (common div/span-based layouts): check the
    // wrapper's own direct text, then the wrapper's preceding sibling.
    let node = el.parentElement;
    let levels = 0;
    while (node && levels < 4 && node !== document.body) {
      const own = ownText(node);
      if (usable(own)) return own;

      const psib = node.previousElementSibling;
      if (psib && !isFieldNode(psib)) {
        const t = textOf(psib);
        if (usable(t)) return t;
      }
      node = node.parentElement;
      levels++;
    }

    return '';
  }

  function fieldsetLegend(el) {
    const fs = el.closest('fieldset');
    if (fs) {
      const legend = fs.querySelector('legend');
      if (legend) return textOf(legend);
    }
    return '';
  }

  let results = [];
  let radioGroups = {};
  let checkboxGroups = {};

  function collect(root) {
    const selector = [
      'input:not([type="hidden"]):not([type="submit"]):not([type="button"]):not([type="reset"]):not([type="image"])',
      'textarea',
      'select',
      '[contenteditable="true"]'
    ].join(',');

    root.querySelectorAll(selector).forEach(el => {
      if (!includeDisabled && (el.disabled || el.getAttribute('aria-disabled') === 'true')) return;
      if (onlyVisible && !isVisible(el)) return;

      const tag = el.tagName.toLowerCase();
      const type = (el.getAttribute('type') ||
                    (tag === 'select' ? 'select' : tag === 'textarea' ? 'textarea' : 'text')).toLowerCase();

      // --- radio buttons: collapse into one logical field per `name` ---
      if (type === 'radio' && el.name) {
        const key = 'radio::' + el.name;
        const opt = { value: el.value, label: resolveLabel(el), checked: el.checked, xpath: getXPath(el) };
        if (radioGroups[key]) {
          radioGroups[key].options.push(opt);
          return;
        }
        const group = {
          index: null, kind: 'radio_group', tag: 'input', type: 'radio',
          name: el.name, label: fieldsetLegend(el) || el.name,
          nearby_text: fieldsetLegend(el) ? '' : nearbyText(el),
          required: el.required || el.getAttribute('aria-required') === 'true',
          xpath: getXPath(el), options: [opt]
        };
        radioGroups[key] = group;
        results.push(group);
        return;
      }

      // --- checkboxes sharing a name (multi-select sets) collapse too ---
      if (type === 'checkbox' && el.name) {
        const siblings = root.querySelectorAll('input[type="checkbox"][name="' + CSS.escape(el.name) + '"]');
        if (siblings.length > 1) {
          const key = 'checkbox::' + el.name;
          const opt = { value: el.value, label: resolveLabel(el), checked: el.checked, xpath: getXPath(el) };
          if (checkboxGroups[key]) {
            checkboxGroups[key].options.push(opt);
            return;
          }
          const group = {
            index: null, kind: 'checkbox_group', tag: 'input', type: 'checkbox',
            name: el.name, label: fieldsetLegend(el) || el.name,
            nearby_text: fieldsetLegend(el) ? '' : nearbyText(el),
            required: false,
            xpath: getXPath(el), options: [opt]
          };
          checkboxGroups[key] = group;
          results.push(group);
          return;
        }
      }

      const entry = {
        index: null,
        kind: tag === 'select' ? 'select' : (type === 'checkbox' ? 'checkbox' : (el.isContentEditable ? 'contenteditable' : 'input')),
        tag: tag,
        type: type,
        label: resolveLabel(el),
        nearby_text: nearbyText(el),
        name: el.getAttribute('name') || '',
        placeholder: el.getAttribute('placeholder') || '',
        required: el.required || el.getAttribute('aria-required') === 'true',
        current_value: tag === 'select' ? '' : (el.isContentEditable ? cleanText(el.innerText) : (el.value || '')),
        maxlength: el.getAttribute('maxlength') || null,
        pattern: el.getAttribute('pattern') || null,
        autocomplete: el.getAttribute('autocomplete') || null,
        xpath: getXPath(el)
      };

      if (tag === 'select') {
        entry.options = Array.from(el.options).map(o => ({ value: o.value, label: cleanText(o.text), selected: o.selected }));
        entry.current_value = (el.options[el.selectedIndex] || {}).value || '';
      }

      results.push(entry);
    });
  }

  collect(document);

  // shadow DOM — walk every element once, recurse into any shadow roots found
  (function walkShadow(root) {
    root.querySelectorAll('*').forEach(el => {
      if (el.shadowRoot) {
        collect(el.shadowRoot);
        walkShadow(el.shadowRoot);
      }
    });
  })(document);

  results.forEach((r, i) => { r.index = i; });
  return results;

})(arguments[0], arguments[1]);
"""


def get_fillable_elements(
    driver,
    only_visible: bool = True,
    include_disabled: bool = False,
    include_iframes: bool = True,
) -> List[Dict[str, Any]]:
    elements = driver.execute_script(_EXTRACT_JS, only_visible, include_disabled)
    for e in elements:
        e["frame_xpath"] = None

    if include_iframes:
        try:
            frames = driver.find_elements(By.TAG_NAME, "iframe")
        except Exception:
            frames = []
        for fi, frame in enumerate(frames):
            try:
                frame_xpath = driver.execute_script(
                    "return arguments[0].id ? '//*[@id=\"'+arguments[0].id+'\"]' : null;", frame
                )
                driver.switch_to.frame(frame)
                inner = driver.execute_script(_EXTRACT_JS, only_visible, include_disabled)
                for e in inner:
                    e["frame_xpath"] = frame_xpath or f"//iframe[{fi + 1}]"
                elements.extend(inner)
            except Exception:
                pass
            finally:
                driver.switch_to.default_content()

    for i, e in enumerate(elements):
        e["index"] = i
    return elements


def get_form_fingerprint(elements: List[Dict[str, Any]]) -> str:
    sig = [(e.get("tag"), e.get("type"), e.get("name"), e.get("label", "").lower().strip()) for e in elements]
    blob = json.dumps(sig, sort_keys=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


class Structure(RootModel[Dict[str, str]]):
    pass


filler_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are filling multiple web form fields.

For every field choose the SINGLE most appropriate value ONLY from the supplied candidates.

Return the structured schema.

Rules:
- Use ONLY values present in the candidates.
- Never invent any information.
- Nearby text is only page context and may be noisy.
- If none of the candidates fit, return an empty string.
- If the field already contains the correct value, return that same value.
- For email fields return only a valid email.
- For phone fields return only the phone number.
- Return answers indexed by the supplied field index.



DO NOT ATTEMPT TO ANSWER THE QUESTIONS THAT YOU ALREADY ANSWERED!!!!!!.
"""
    ),
    MessagesPlaceholder("history"),    
    (
        "human",
        """
Below are all the fields that require disambiguation.

{fields}
"""
    )
])


from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

session={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    print("session_history called")
    if session_id not in session:
        print ("New session created....")
        session[session_id]=ChatMessageHistory()
    return session[session_id]


already_answered=[]
class CachedFieldMapper:

    def __init__(self, llm, sim_search):
        self.llm = llm
        self.sim_search = sim_search

        structured_model = self.llm.with_structured_output(Structure)

        self.chain = filler_prompt | structured_model


    def match(
        self,
        elements: List[Dict[str, Any]],
        catalogue: Dict[str, str]
    ) -> Dict[int, str]:

        clean_catalogue = {
            k: str(v)
            for k, v in catalogue.items()
            if v is not None and str(v).strip()
        }

        if not clean_catalogue:
            return {}

        index_label_type_cur = [
            (
                e["index"],
                (
                    e.get("label")
                    or e.get("name")
                    or e.get("placeholder")
                    or e.get("nearby_text")
                    or ""
                ).strip(),
                e.get("type", "text"),
                str(e.get("current_value") or ""),
                (e.get("nearby_text") or "").strip()
            )
            for e in elements
            if (
                e.get("label")
                or e.get("name")
                or e.get("placeholder")
                or e.get("nearby_text")
                or ""
            ).strip()
        ]

        if not index_label_type_cur:
            return {}

        labels = [x[1] for x in index_label_type_cur]
        cat_keys = list(clean_catalogue.keys())

        query_embeddings = self.sim_search.encode(labels)
        key_embeddings = self.sim_search.encode(cat_keys)

        scores = cosine_similarity(query_embeddings, key_embeddings)

        returning_dict = {}

        llm_fields = []

        # ---------------- Similarity Search ---------------- #

        for i, (idx, label, ftype, cur_val, nearby_text) in enumerate(index_label_type_cur):

            ranked = sorted(
                [
                    (
                        cat_keys[j],
                        clean_catalogue[cat_keys[j]],
                        float(scores[i][j])
                    )
                    for j in range(len(cat_keys))
                ],
                key=lambda x: x[2],
                reverse=True
            )

            # High confidence -> no LLM needed
            if ranked[0][2] > 0.92:
                returning_dict[idx] = ranked[0][1]
                continue

            llm_fields.append({
                "index": idx,
                "label": label,
                "field_type": ftype,
                "current_value": cur_val,
                "nearby_text": nearby_text,
                "candidates": ranked[:5]
            })

        # Nothing left for LLM
        if not llm_fields:
            return returning_dict

        # ---------------- Batched LLM Calls ---------------- #
        # Everything used to go into ONE giant prompt with every ambiguous field
        # at once. That gave the model a lot to reason about in a single turn,
        # which is exactly what burned through the token budget and produced
        # `BadRequestError: output_parse_failed` (the model's chain-of-thought
        # never finished before it was supposed to emit the structured answer).
        # Batching keeps each call small, and a single bad batch no longer takes
        # the rest of the fields down with it.

        BATCH_SIZE = 8

        history_chain = RunnableWithMessageHistory(
            self.chain,
            get_session_history,
            input_messages_keys="fields",
            history_messages_key="history"
        )

        for batch_start in range(0, len(llm_fields), BATCH_SIZE):
            batch = llm_fields[batch_start:batch_start + BATCH_SIZE]

            prompt_text = ""

            for field in batch:

                candidate_text = "\n".join(
                    f"{k}: {v}"
                    for k, v, _ in field["candidates"]
                )

                prompt_text += f"""
======================================================
Index: {field['index']}

Field:
{field['label']}

Type:
{field['field_type']}

Current Value:
{field['current_value']}

Nearby Text:
{field['nearby_text']}

Candidates:
{candidate_text}

======================================================


DO NOT ATTEMPT THE QUESTIONS THAT YOU HAVE ALREADY ANSWERED (LOOK INTO YOUR CHAT HISTORY)!!!!!

"""

            print("Previous session: ", session)
            pre = len(session)

            try:
                response = history_chain.invoke({
                    "fields": prompt_text
                }, config={"configurable": {"session_id": "chat1"}})
            except Exception as e:
                # Don't let one bad batch (a Groq hiccup, a truncated reasoning
                # trace, a rate limit, etc.) crash the whole fill. Keep whatever
                # the similarity search already resolved and move on to the
                # next batch.
                print(f"[CachedFieldMapper] batch starting at field "
                      f"{batch[0]['index']} failed, skipping it: {e}")
                continue

            print("Updated_session: ", session)
            print("Difference i session: ", len(session) - pre)

            # ---------------- Parse Output ---------------- #

            for idx, value in response.root.items():

                value = value.strip()

                if value and value.lower() not in (
                    "none",
                    "na",
                    "n/a",
                    "-none-",
                    ""
                ):
                    returning_dict[int(idx)] = value

        return returning_dict

visited=[]
def fill_elements(
    driver,
    elements: List[Dict[str, Any]],
    mapping: Dict[int, str],
    catalogue: Dict[str, str] = None,
    timeout: int = 10,
) -> Dict[str, Any]:
    by_index = {e["index"]: e for e in elements}
    results = {"filled": [], "skipped": [], "failed": []}
    wait = WebDriverWait(driver, timeout)

    for idx, value in mapping.items():
        if by_index.get(idx) not in visited:
          visited.append(by_index.get(idx))
          el = by_index.get(idx)
          if el is None or not str(value).strip():
              results["skipped"].append({"index": idx, "reason": "no element or empty value"})
              continue

          if el.get("frame_xpath"):
              try:
                  frame = driver.find_element(By.XPATH, el["frame_xpath"])
                  driver.switch_to.frame(frame)
              except Exception:
                  pass

          try:
              if el["type"] == "file":
                results["skipped"].append({"index": idx, "reason": "file input"})
                node = wait.until(EC.presence_of_element_located((By.XPATH, el["xpath"])))
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", node)
                dropdown = driver.find_element(By.XPATH, el["xpath"])

                dropdown.click()
                dropdown.send_keys("C:/Users/rohit/Desktop/All_Language_programs/resume.pdf")
                dropdown.send_keys(Keys.ENTER)

              elif el["kind"] in ("radio_group", "checkbox_group"):
                  target = None
                  value_l = str(value).strip().lower()
                  for opt in el["options"]:
                      if value_l in (opt["label"].lower(), str(opt["value"]).lower()):
                          target = opt
                          break
                  if target:
                      node = wait.until(EC.element_to_be_clickable((By.XPATH, target["xpath"])))
                      if not node.is_selected():
                          driver.execute_script("arguments[0].scrollIntoView({block:'center'});", node)
                          node.click()
                      results["filled"].append({"index": idx, "value": value})
                  else:
                      results["skipped"].append({"index": idx, "reason": f"no matching option for '{value}'"})

              elif el["kind"] == "select":

                node = wait.until(EC.presence_of_element_located((By.XPATH, el["xpath"])))
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", node)
                dropdown = driver.find_element(By.XPATH, el["xpath"])
                dropdown.click()
                value_l=str(value).strip().lower()
                picked=True


                list_options=[opt["label"].lower() or str(opt["value"]).lower() for opt in el.get("options")]

                encoded_value_l=sim_search.encode([value_l])
                encoded_list_options=sim_search.encode(list_options)


                print("encoded value l: ", value_l, "encoded_list_options: ", list_options)

                scores=cosine_similarity(encoded_value_l, encoded_list_options)

                option=list_options[np.argmax(scores)]

                print ("selected opt: ", option)
                dropdown.send_keys(option)

                dropdown.send_keys(Keys.DOWN)
                dropdown.send_keys(Keys.ENTER)
                # for opt in el.get("options"):
                #     if value_l in (opt["label"].lower(), str(opt["value"]).lower()):
                #         if value_l.isnumeric():
                #             dropdown.send_keys(int(opt["label"].lower() or str(opt["value"]).lower()))
                #         else:
                #             dropdown.send_keys(opt["label"].lower() or str(opt["value"]).lower())
                # #dropdown.send_keys(value)
                #         dropdown.send_keys(Keys.DOWN)
                #         dropdown.send_keys(Keys.ENTER)
                #         picked=True
                #         break

                if picked:
                     results["filled"].append({"index": idx, "value": value})
                else:
                     results["skipped"].append({"index": idx, "reason": f"no matching select option for '{value}'"})


              #   node = wait.until(EC.presence_of_element_located((By.XPATH, el["xpath"])))
              #   driver.execute_script("arguments[0].scrollIntoView({block:'center'});", node)
              #   from selenium.webdriver.support.ui import Select
              #   sel = Select(node)
              #   value_l = str(value).strip().lower()
              #   picked = False
              #   for opt in el.get("options", []):
              #       if value_l in (opt["label"].lower(), str(opt["value"]).lower()):
              #           sel.select_by_value(opt["value"])
              #           picked = True
              #           break
              #   if picked:
              #       results["filled"].append({"index": idx, "value": value})
              #   else:
              #       results["skipped"].append({"index": idx, "reason": f"no matching select option for '{value}'"})

              else:

                  def _native_set_value(node, value):
                      # Fallback for React/Vue/Angular-controlled inputs: send_keys can be
                      # silently dropped if the framework re-renders faster than Selenium
                      # is typing, or if the field filters out characters as they're typed
                      # (very common on type="number" inputs). This sets the value through
                      # the native property setter and fires the events the framework
                      # actually listens for, so the UI state updates for real.
                      driver.execute_script("""
                          const node = arguments[0];
                          const value = arguments[1];
                          const tag = node.tagName.toLowerCase();
                          const proto = tag === 'textarea'
                              ? window.HTMLTextAreaElement.prototype
                              : window.HTMLInputElement.prototype;
                          const setter = Object.getOwnPropertyDescriptor(proto, 'value').set;
                          setter.call(node, value);
                          node.dispatchEvent(new Event('input', {bubbles: true}));
                          node.dispatchEvent(new Event('change', {bubbles: true}));
                      """, node, value)

                  def decide_and_send_keys(node, value):
                      value_str = str(value).strip()
                      is_number_field = (el.get("type") in ("number", "tel")) or \
                          value_str.replace(".", "", 1).replace("-", "", 1).isdigit()

                      if is_number_field:
                          # Strip commas / currency symbols / stray spaces. If that strips
                          # everything away, this wasn't really a numeric value to begin
                          # with (e.g. a mismatch put "Not disclosed" here) — type the raw
                          # string rather than silently sending an empty send_keys("").
                          clean_value = re.sub(r"[^\d.\-]", "", value_str) or value_str
                          node.send_keys(clean_value)
                          time.sleep(0.3)
                          # IMPORTANT: only step in with the native-setter fallback if the
                          # field is still genuinely EMPTY. Do NOT compare against
                          # clean_value for equality — masked/formatted money fields
                          # legitimately redisplay "20,00,000" or "₹2000000" after a
                          # successful type, which will never string-match clean_value.
                          # Firing the native-setter overwrite on every non-exact-match
                          # was fighting the page's own formatting logic and is what left
                          # the field blank after the last patch.
                          if not (node.get_attribute("value") or "").strip():
                              _native_set_value(node, clean_value)
                              time.sleep(0.2)
                      else:
                          node.send_keys(value_str)

                  node = wait.until(EC.presence_of_element_located((By.XPATH, el["xpath"])))
                  driver.execute_script("arguments[0].scrollIntoView({block:'center'});", node)
                  if el["kind"] == "contenteditable":
                      driver.execute_script("arguments[0].innerText = '';", node)
                      decide_and_send_keys(node, value)
                      final_value = driver.execute_script("return arguments[0].innerText;", node)
                  else:
                      node.clear()
                      decide_and_send_keys(node, value)
                      final_value = node.get_attribute("value")

                  # A blind Keys.ENTER here risked submitting the form early or resetting
                  # the field before the typed value "stuck" on some layouts. Tab out
                  # instead — it still fires blur/validation without that risk.
                  node.send_keys(Keys.TAB)

                  # Only report success if something actually landed in the field —
                  # previously this was reported as "filled" unconditionally, which is
                  # why silent failures were invisible in `results`.
                  if str(final_value or "").strip():
                      already_answered.append(value)
                      results["filled"].append({"index": idx, "value": value})
                  else:
                      results["failed"].append({"index": idx, "error": "field still empty after fill attempt"})

          except Exception as e:
              results["failed"].append({"index": idx, "error": str(e)})
              continue
          finally:
              if el.get("frame_xpath"):
                  driver.switch_to.default_content()

          history=get_session_history("chat1")
          history.add_ai_message(f"Okay I have already answered: {', '.join(already_answered)}")
        else:
          continue
    return results


#DRIVER = None




In [ ]:


driver_path = r"chromedriver.exe"

service = Service(driver_path)
driver = webdriver.Chrome(service=service)

global_image_path_for_ss="test_image_path.png"
driver_path = r"chromedriver.exe"

history_catalogue=[]



@tool
def scroll_page(reason_to_scroll_page: str)-> None:
    """Use this tool to scroll the page"""

    driver.execute_script("window.scrollBy(0, window.innerHeight);")
    
def flatten_to_clean_dict(nested_dict, parent_key='', sep='_'):
    flat_data = {}
    
    for key, value in nested_dict.items():
        # Build the structured string key
        new_key = f"{parent_key}{sep}{key}" if parent_key else key
        
        # 1. Handle nested dictionaries
        if isinstance(value, dict):
            flat_data.update(flatten_to_clean_dict(value, new_key, sep=sep))
            
        # 2. Handle lists
        elif isinstance(value, list):
            # If it's a list of dictionaries (like education, projects)
            if all(isinstance(i, dict) for i in value):
                for index, item in enumerate(value):
                    flat_data.update(flatten_to_clean_dict(item, f"{new_key}_{index}", sep=sep))
            else:
                # If it's a list of strings/items (like skills, achievements), join with commas
                flat_data[new_key] = ", ".join(map(str, value))
                
        # 3. Handle primitive data types (Strings, Ints, Booleans)
        else:
            flat_data[new_key] = str(value) if isinstance(value, bool) else value
                
    return flat_data

import json
with open('resume.json', 'r') as file:
    data_dict = json.load(file)


flat_resume = flatten_to_clean_dict(data_dict)

service = Service(driver_path)
@tool
def fill_input_fields(field, value, what_have_you_done):
    """
    Find a text input or textarea on the page and type a value into it.

    Use this tool whenever you need to fill any input field automatically,
    including but not limited to:
    - Name, email, phone number fields
    - Address, city, state, zip fields
    - Cover letter or essay text areas
    - Any open-ended or short-answer question field
    - Search boxes that require typed input
    - Any field where text must be entered programmatically

    The argument must be a JSON string with two keys:
    - "field": a natural language description of the input field (its label, placeholder, or purpose)
    - "value": the exact text to type into that field

    Example usage:
        fill_input_fields('{"field": "First name", "value": "Rohith"}')
        fill_input_fields('{"field": "Email address", "value": "rohit@email.com"}')
        fill_input_fields('{"field": "Why do you want to work here?", "value": "I am passionate about..."}')

    Do NOT use this tool for:
    - Clicking buttons or links (use find_text_based_element_index instead)
    - Clicking checkboxes (use click_checkboxes instead)
    - Any action that does not require typing text

    Args:
        field_description_and_value: A JSON string with keys "field" and "value".
        what_have_you_done: what has been done in this stage.
    """
    # import test as fet
    
    # fet.DRIVER = driver

    # add fet.list_user_fillable_elements to your `tools` list so the
    # agent can call it directly, OR drive it yourself for bulk runs:

    elements = get_fillable_elements(driver)

    mapper = CachedFieldMapper(llm=model, sim_search=sim_search)   # reuse your existing ChatGroq model
    catalogue_row = {"first_name": "Rohit", "last_name": "Kumar", "email": "r74892931@gmail.com", "phone_number":"1234567891", "date_of_birth":'31', "month_of_birth":"January", "year_of_birth":"2005", "Gender":"Male", "dob":"January 31 2005"}
    mapping = mapper.match(elements, flat_resume)



    results = fill_elements(driver, elements, mapping, catalogue_row)
    print(results)
    def click_all_checkboxes(driver, timeout=10):
        results = {"clicked": 0, "already_checked": 0, "failed": 0, "total_found": 0}

        native_checkboxes = driver.find_elements(By.CSS_SELECTOR, "input[type='checkbox']")
        print(f"Found {len(native_checkboxes)} native checkboxes")

        for cb in native_checkboxes:
            results["total_found"] += 1
            try:
                if not cb.is_selected():
                    # Scroll into view first
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", cb)
                    time.sleep(0.2)

                    try:
                        cb.click()
                    except ElementClickInterceptedException:
                        # Fallback: JS click (handles overlapping elements)
                        driver.execute_script("arguments[0].click();", cb)
                    results["clicked"] += 1
                else:
                    print(f" Already checked: id='{cb.get_attribute('id') or 'N/A'}'")
                    results["already_checked"] += 1

            except StaleElementReferenceException:
                print("Stale element — page may have re-rendered, skipping.")
                results["failed"] += 1
            except ElementNotInteractableException:
                # Try JS click as last resort
                try:
                    driver.execute_script("arguments[0].click();", cb)
                    results["clicked"] += 1
                except Exception as e:
                    print(f"No interaction {e}")
                    results["failed"] += 1
            except Exception as e:
                print(f"{e}")
                results["failed"] += 1
        aria_checkboxes = driver.find_elements(By.CSS_SELECTOR, "[role='checkbox']")
        print(f"Found {len(aria_checkboxes)} ARIA-role checkboxes")

        for cb in aria_checkboxes:
            results["total_found"] += 1
            try:
                is_checked = cb.get_attribute("aria-checked")
                if is_checked != "true":
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", cb)
                    time.sleep(0.2)
                    try:
                        cb.click()
                    except ElementClickInterceptedException:
                        driver.execute_script("arguments[0].click();", cb)
                    results["clicked"] += 1
                else:
                    results["already_checked"] += 1

            except Exception as e:
                print(f"{e}")
                results["failed"] += 1
        shadow_clicked = driver.execute_script("""
            let count = 0;
            function clickInShadow(root) {
                let boxes = root.querySelectorAll("input[type='checkbox'], [role='checkbox']");
                boxes.forEach(cb => {
                    if (!cb.checked && cb.getAttribute('aria-checked') !== 'true') {
                        cb.click();
                        count++;
                    }
                });
                root.querySelectorAll('*').forEach(el => {
                    if (el.shadowRoot) clickInShadow(el.shadowRoot);
                });
            }
            clickInShadow(document);
            return count;
        """)
        if shadow_clicked:
            print(f"✅ Clicked {shadow_clicked} Shadow DOM checkboxes")
            results["clicked"] += shadow_clicked
            results["total_found"] += shadow_clicked

        return results
    click_all_checkboxes(driver)

    if str(input("Was all the fields been inputted?:"))=="y":
        

        os.remove(global_image_path_for_ss)
        driver.save_screenshot(global_image_path_for_ss)
        history_catalogue.append(what_have_you_done)




    
@tool
def find_visual_element_index(image_button_to_be_clicked:str, what_have_you_done:str)->None:
    """
    Click a webpage element that can ONLY be identified by how it looks visually —
    not by any readable text, label, placeholder, or aria attribute.

    Use this tool exclusively when the target element:
    - Has no visible text, label, or accessible name (e.g. a bare icon, logo, or symbol)
    - Is identified by its shape, color, position, or graphic (e.g. a hamburger menu,
      a magnifying-glass icon, a close "×" button with no text, a star rating widget)
    - Is a purely decorative or symbolic control where meaning comes from the image itself

    Do NOT use this tool when:
    - The element has any readable text (e.g. "Submit", "Next", "Accept All Cookies")
    - The element has an aria-label, placeholder, or tooltip that names it
    - You can describe the target by what it says rather than what it looks like

    Examples of correct usage:
      "Click the hamburger menu icon in the top-left corner"
      "Click the magnifying glass search icon"
      "Close the popup using the X icon button"
      "Click the gear/settings icon"

    Examples where find_text_based_element_index should be used instead:
      "Click the Accept button"         ← has visible text
      "Click the Search input field"    ← has a placeholder
      "Click the navigation menu"       ← has an aria-label

    Args:
        image_button_to_be_clicked: A concise visual description of the target element,
            focusing on its appearance, symbol, shape, or location on screen.
        what_have_you_done: what has been done in this stage.
    x: int
    y: int
    image_path: str
    driver: Any
    to_be_found: str
    button_selector_vlm_model: Any
    """
    output=app.invoke({
        "driver": driver,
        "to_be_found":image_button_to_be_clicked,
        "image_path":global_image_path_for_ss,
        "button_selector_vlm_model": button_selector_vlm_model
    })

    print(output)
    os.remove(global_image_path_for_ss)
    driver.save_screenshot(global_image_path_for_ss)
    history_catalogue.append(what_have_you_done)
    return "Done clicking the icon"

class text_based_element_structure(BaseModel):
    index_number: str=Field(description="The index number of the text button to be clicked")

@tool
def find_text_based_element_index(button_to_be_clicked:str, what_have_you_done:str)->None:
    """
    Click a webpage element that can be identified by its readable text, label,
    placeholder, or accessible name — without needing to inspect the page visually.

    Use this tool when the target element has ANY of the following:
    - Visible button or link text (e.g. "Sign In", "Accept All Cookies", "Next Step")
    - An input placeholder (e.g. a search box labeled "Search...", "Enter email")
    - An aria-label or accessible name (e.g. a button labeled "Close dialog")
    - A dropdown or menu item with a readable title

    Do NOT use this tool when:
    - The element has no text whatsoever and is identified purely by its icon or image
    - The target is a symbol (hamburger menu, ×, ★, ♥) with no associated text
    - You would need to look at the rendered page to recognize the element

    Examples of correct usage:
      "Click the Accept All Cookies button"
      "Click the Sign In link"
      "Click the Search input field"
      "Select the dropdown option labeled Monthly"
      "Close the newsletter popup"

    Examples where find_visual_element_index should be used instead:
      "Click the hamburger icon"    ← icon only, no text
      "Click the X to close"        ← symbol only
      "Click the settings gear"     ← icon only

    Args:
        button_to_be_clicked: A natural language description of the intended action
            or target element, referencing its visible text or label.
        what_have_you_done: What has been done in this stage.
    """
    prompt=ChatPromptTemplate.from_messages([("system", """
You are an element selection engine.

Your task is to choose exactly ONE element from the provided list.

You are given:

1. User Intent
2. Available interactive elements.

Each element contains:
- index
- type
- visible_text
- aria_label (optional)
- placeholder (optional)

Selection Procedure:

1. Understand the user's intent.
2. Compare the intent against EVERY element.
3. Select the element whose purpose best satisfies the intent.
4. Prefer exact text matches over semantic matches.
5. Prefer buttons over links only if both perform the same action.
6. Ignore decorative or unrelated elements.
7. Never invent an index.
8. Return exactly one index from the list.

Priority Order:

1. Exact visible text match
2. Exact aria-label match
3. Exact placeholder match
4. Strong semantic match
5. Partial semantic match

If multiple elements match:

- Prefer the most specific wording.
- Prefer action buttons over navigation links.
- Prefer primary actions over secondary actions.


Return the index nothing else.
"""), ("human", """User Intent:
{explanation}

Available Elements:
{buttons}""")])
    # driver.execute_script("""
    # let overlay=document.getElementById('relyance-banner-container');
    # if(overlay){
    #     overlay.remove();
    # }
    # """)
    
    answer=get_interactive_elements(driver)
    button_data = []

    for ele in answer:
        text = ele.get("text") or ele.get("aria_label") or ele.get("placeholder")

        if text and text.strip():
            button_data.append({
                "index": ele["index"],
                "text": text,
                "type": ele["type"],
                "xpath": ele["xpath"]
            })
    button_texts = [x["text"] for x in button_data]

    query_embedding = sim_search.encode([button_to_be_clicked])
    button_embeddings = sim_search.encode(button_texts)

    scores = cosine_similarity(query_embedding, button_embeddings)[0]

    ranked = sorted(
        zip(button_data, scores),
        key=lambda x: x[1],
        reverse=True
    )

    TOP_K = 10

    candidate_elements = [element for element, _ in ranked[:TOP_K]]


    print ("condidate_elements: ",candidate_elements)
    structured_model=small_model.with_structured_output(text_based_element_structure) #-<<<

    chain = prompt | structured_model

    response = chain.invoke({
        "explanation": button_to_be_clicked,
        "buttons": json.dumps(candidate_elements, indent=2)
    })

    print(response)

    selected_index = int(response.index_number)

    selected_element = next(
        ele for ele in answer
        if ele["index"] == selected_index
    )

    print("Selected Element:")
    print(selected_element)

    xpath = selected_element["xpath"]

    safe_click(driver, xpath)

    os.remove(global_image_path_for_ss)
    driver.save_screenshot(global_image_path_for_ss)
    history_catalogue.append(what_have_you_done)

    return "DONE CLICKING"





    
tools=[find_text_based_element_index, find_visual_element_index,  fill_input_fields] #click_checkboxes,
from langchain.chains.base import Chain

class ss_custom_chain(Chain):
    @property
    def input_keys(self):
        return ["path", "his"]

    @property
    def output_keys(self):
        return ["base64_image"]

    def _call(self, inputs, run_manager = None):
        print("saving_sss")
        driver.save_screenshot(inputs["path"])
        base64_image=encode_image(inputs["path"])
        return {"base64_image":base64_image, "history":inputs["his"]}



ss_runnable2=ss_custom_chain()
prompt=ChatPromptTemplate.from_messages([("system", """You are an intelligent web navigation agent.

You must solve every task by calling one of the available tools.

Do not answer in natural language.

Whenever a field needs to be filled, call fill_input_fields.

Whenever a button needs to be clicked, call find_text_based_element_index or find_visual_element_index.

Only produce tool calls.

Do not assume information that is not visible.

Use the previous observations and actions in the scratchpad to maintain context across steps.

Previous observations and actions:

"""), ("human", [{
        "type":"text",
        "text": "Apply for this Job position by taking necessary actions, My user name is Rohithkumar2006 and password is indian@2014"
    },
    {"type":"image_url", "image_url": {
        "url":"{base64_image}"}
    }]),("human", "Here are the following procedures which has already been done, do not perform them again and again: \n History: {history}"), MessagesPlaceholder("agent_scratchpad")])


init=create_tool_calling_agent(
    llm=vlm_model,
    tools=tools,
    prompt=prompt
)
agent=AgentExecutor(tools=tools, agent=init, verbose=True, max_iterations=1)


from langchain_core.runnables import RunnableLambda
chain=ss_runnable2 | agent



driver.get("https://smartinterviews.in/placements/job/1538")

for i in range(20):
    print(f"--- Iteration {i}: perform action if needed ---")
    time.sleep(2)
    vision_inter_output=chain.invoke({
                "path": global_image_path_for_ss,
                "his":" ".join(map(str, history_catalogue))
            })

            
    


--- Iteration 0: perform action if needed ---


In [59]:
a=["a", "b"]

print(" ".join(map(str, a)))

a b
